<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-11-cost-and-performance/lesson-11.4-latency-optimization/notebooks/exercises-11.4.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 11.4 — Latency Optimization
Netsetos GenAI Course v2.0 | Module 11: Cost & Performance

TTFT vs TPOT, streaming, prompt cache, geo routing, speculative decoding, continuous batching, tail mitigations.


In [ ]:
print('Module 11: Cost & Performance')
print('Lesson 11.4: Latency Optimization')


## Ex 1: Latency Anatomy


In [ ]:
print('Latency components for a typical RAG call:')
for component, ms_typical, knob in [
    ('Prompt build + tokenize',     5, 'minor'),
    ('Network RTT (US-East)',      250,'geo route'),
    ('Provider queue + TTFT',     1200,'prompt cache'),
    ('Retrieval (vector DB)',      40, 'index tuning'),
    ('Generation (500 out tok)',  3500,'speculative + smaller model'),
    ('Parse + post-hooks',         15, 'minor'),
]:
    print(f'  {component:32s} ~{ms_typical:>5} ms | knob: {knob}')
print()
print('Total ~5 seconds. Generation dominates; TTFT is what user feels.')


## Ex 2: TTFT vs Total Time — Perceived Speed


In [ ]:
print('Perceived speed by TTFT (same total wall time):')
for ttft_ms, total_ms, perceived in [
    (4000, 4000, 'feels broken (non-streaming)'),
    (1200, 4000, 'noticeable lag'),
    ( 280, 4000, 'feels fast (5x perceived)'),
    ( 100, 4000, 'feels instant'),
]:
    print(f'  TTFT={ttft_ms:>4}ms  total={total_ms}ms -> {perceived}')


## Ex 3: Geo Routing Impact (India)


In [ ]:
print('India RTT to provider region:')
for region, rtt_ms, deploys in [
    ('US-East (default)',       260, 'OpenAI/Anthropic default'),
    ('US-West',                 220, 'fallback'),
    ('Singapore (asia-south)',   65, 'Anthropic / OpenAI region'),
    ('Mumbai (asia-south1)',      8, 'E2E, Yotta, Cloud Run, Vertex AI'),
]:
    print(f'  {region:30s} ~{rtt_ms:>3}ms | {deploys}')
print()
print('Mumbai cuts ~250ms per call vs US-East. Free win for Indian apps.')


## Ex 4: Prompt Cache TTFT Cut


In [ ]:
print('Prompt cache effect on TTFT (Anthropic, 4k system prompt):')
for state, ttft_ms, gen_unaffected in [
    ('First call (creation)',  1300, 'gen still 3.5s'),
    ('Cached read (within 5min)',280, 'gen still 3.5s'),
    ('Cached read (1h cache)',   280, 'gen still 3.5s'),
    ('Cache miss (cold)',       1300, 'gen still 3.5s'),
]:
    print(f'  {state:32s}: TTFT {ttft_ms}ms | {gen_unaffected}')
print()
print('Prompt cache saves TTFT, not generation. TTFT is what users feel.')


## Ex 5: Speculative Decoding


In [ ]:
print('Speculative decoding speedup (vLLM):')
for setup, k, accept_rate, speedup in [
    ('No spec',           '-',  '-',     '1.0x'),
    ('Llama-1B draft',     5,  '0.65',   '2.1x'),
    ('Llama-1B draft',     8,  '0.55',   '2.5x'),
    ('Eagle/Medusa draft', 8,  '0.78',   '3.0x'),
]:
    print(f'  {setup:22s} K={k:3s} accept={accept_rate:5s} -> {speedup} TPOT')
print()
print('Best when draft model is often right (boilerplate, JSON, code).')


## Ex 6: Output Discipline Saves Wall Time


In [ ]:
print('Generation time = output_tokens / TPOT. Cut output, cut wall time:')
tpot_ms = 7  # ms per token
for label, out_tok in [
    ('free-text response',    800),
    ('+ max_tokens=500',      500),
    ('+ JSON schema',         300),
    ('+ bullet cap (3 max)',  180),
    ('+ specialist FT model', 180),  # output same, but faster TPOT
]:
    wall = out_tok * tpot_ms
    print(f'  {label:28s} out={out_tok:>4} tok -> {wall:>4} ms generation')


## Ex 7: Tail Mitigation (p99 Flattening)


In [ ]:
print('p99 latency drivers and mitigations:')
for cause, impact, fix in [
    ('Provider hiccup',        'p99 = 8s spike', 'fallback provider, hedge'),
    ('Agent loop runaway',     'p99 unbounded',   'max_turns + budget cap'),
    ('Long prompt outlier',    'p99 5x p50',     'prompt size cap + chunk'),
    ('Cold start (Cloud Run)', 'p99 +5s',         'min-instances > 0'),
    ('Queue saturation',       'p99 unbounded',   'front-door shedding'),
]:
    print(f'  {cause:24s}: {impact:18s} | fix: {fix}')


---
## Recap
TTFT < 500ms target. Stream anything > 500ms. p99 matters as much as p50.
Stack: SSE streaming + prompt cache (50-80% TTFT cut) + geo routing (-250ms India) +
speculative decoding (2-3x TPOT) + output discipline + tail mitigations.
Profile with OTel before optimizing — the dominant cost is rarely where you think.
